# ST1 — DBT Gold Setup (Google Colab)
**Projet RFM CLV E-Commerce**

Ce notebook :
1. Monte votre Google Drive
2. Détecte l'état de DBT_Gold (vide ou initialisé)
3. Crée toute la structure dbt + modèles SQL ST1
4. Lance `dbt run` + `dbt test`

> **Relancez les cellules dans l'ordre à chaque nouvelle session Colab.**

## Cellule 1 — Monter Google Drive

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ============================================================
# ADAPTEZ CE CHEMIN si votre projet n'est pas à la racine
# ============================================================
PROJECT_PATH = "/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce"
DBT_PATH     = f"{PROJECT_PATH}/DBT_Gold"
DATA_PATH    = f"{PROJECT_PATH}/Data"
SILVER_PATH  = f"{DATA_PATH}/silver"
GOLD_DB      = f"{DATA_PATH}/gold/gold.duckdb"

# Vérification
assert os.path.exists(PROJECT_PATH), f"Projet introuvable : {PROJECT_PATH}"
assert os.path.exists(SILVER_PATH),  f"Silver introuvable : {SILVER_PATH}"

print("Drive monté")
print("Projet    :", PROJECT_PATH)
print("Silver    :", os.listdir(SILVER_PATH))
print("DBT_Gold  :", os.listdir(DBT_PATH) if os.listdir(DBT_PATH) else "[VIDE — sera initialisé]")

Mounted at /content/drive
Drive monté
Projet    : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce
Silver    : ['ecommerce_shipping_silver.parquet', 'fact_returns.parquet', 'online_retail_silver.parquet']
DBT_Gold  : ['dbt_project.yml', 'target', 'models', 'logs']


## Cellule 2 — Installer dbt-duckdb
> À relancer à chaque session Colab (environnement éphémère).

In [3]:
!pip install dbt-core dbt-duckdb duckdb --quiet
!dbt --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.7/442.7 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.7/238.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44

## Cellule 3 — Créer profiles.yml
> Le `profiles.yml` va dans `~/.dbt/` (jamais dans Drive).

In [4]:
import os

profiles_dir = os.path.expanduser("~/.dbt")
os.makedirs(profiles_dir, exist_ok=True)
os.makedirs(f"{DATA_PATH}/gold", exist_ok=True)

profiles_content = f"""ecommerce:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: "{GOLD_DB}"
      threads: 4
"""

with open(f"{profiles_dir}/profiles.yml", "w") as f:
    f.write(profiles_content)

print("profiles.yml créé dans", profiles_dir)
print(f"DuckDB output : {GOLD_DB}")

profiles.yml créé dans /root/.dbt
DuckDB output : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/gold.duckdb


## Cellule 4 — Initialiser la structure DBT_Gold

In [5]:
import os

# Créer les dossiers nécessaires
folders = [
    "models/staging",
    "models/marts",
    "tests"
]
for folder in folders:
    os.makedirs(f"{DBT_PATH}/{folder}", exist_ok=True)
    print(f"Créé : DBT_Gold/{folder}")

# dbt_project.yml
dbt_project_yml = """name: ecommerce
version: '1.0.0'
profile: ecommerce

model-paths: [\"models\"]
test-paths: [\"tests\"]

models:
  ecommerce:
    staging:
      materialized: view
    marts:
      materialized: table
"""

with open(f"{DBT_PATH}/dbt_project.yml", "w") as f:
    f.write(dbt_project_yml)
print("\ndbt_project.yml écrit")

Créé : DBT_Gold/models/staging
Créé : DBT_Gold/models/marts
Créé : DBT_Gold/tests

dbt_project.yml écrit


## Cellule 5 — Écrire les modèles Staging

In [6]:
# ── stg_transactions.sql ──────────────────────────────────────
stg_transactions = f"""select
    InvoiceNo                                       as invoice_no,
    InvoiceDate                                     as invoice_date,
    invoice_date_only                               as session_date,
    invoice_year,
    invoice_month,
    invoice_week,
    invoice_day_of_week,
    invoice_hour,
    CustomerID                                      as customer_id,
    StockCode                                       as stock_code,
    Description                                     as description,
    upper(split_part(trim(Description), ' ', 1))    as category_proxy,
    Quantity                                        as quantity,
    UnitPrice                                       as unit_price,
    TotalRevenue                                    as line_revenue,
    Country                                         as country,
    case
        when UnitPrice < 2   then 'Low (<2 GBP)'
        when UnitPrice < 10  then 'Medium (2-10)'
        else                      'High (>10)'
    end                                             as price_tier
from read_parquet('{SILVER_PATH}/online_retail_silver.parquet')
where is_identified     = true
  and is_return         = false
  and Quantity          > 0
  and is_quantity_outlier = false
"""

# ── stg_returns.sql ───────────────────────────────────────────
stg_returns = f"""select
    InvoiceNo               as invoice_no,
    StockCode               as stock_code,
    Description             as description,
    abs(Quantity)           as qty_returned,
    InvoiceDate             as invoice_date,
    UnitPrice               as unit_price,
    CustomerID              as customer_id,
    Country                 as country,
    abs(TotalRevenue)       as revenue_lost
from read_parquet('{SILVER_PATH}/fact_returns.parquet')
where CustomerID is not null
"""

# ── stg_shipping.sql ──────────────────────────────────────────
stg_shipping = f"""select
    order_id,
    mode_of_shipment,
    weight_gms,
    weight_category,
    delivered_on_time,
    delivery_status,
    customer_rating,
    satisfaction_label,
    prior_purchases,
    customer_care_calls,
    product_importance,
    cost_of_product,
    discount_pct,
    discount_category,
    estimated_shipping_cost,
    margin_proxy
from read_parquet('{SILVER_PATH}/ecommerce_shipping_silver.parquet')
"""

# Écriture sur Drive
files = {
    "models/staging/stg_transactions.sql" : stg_transactions,
    "models/staging/stg_returns.sql"      : stg_returns,
    "models/staging/stg_shipping.sql"     : stg_shipping,
}
for path, content in files.items():
    with open(f"{DBT_PATH}/{path}", "w") as f:
        f.write(content)
    print(f"Écrit : {path}")

Écrit : models/staging/stg_transactions.sql
Écrit : models/staging/stg_returns.sql
Écrit : models/staging/stg_shipping.sql


## Cellule 6 — Écrire les modèles Marts (ST1)

In [7]:
# ── fact_transactions.sql ─────────────────────────────────────
fact_transactions = """select
    invoice_no,
    invoice_date,
    session_date,
    invoice_year,
    invoice_month,
    invoice_week,
    invoice_day_of_week,
    invoice_hour,
    customer_id,
    stock_code,
    description,
    category_proxy,
    quantity,
    unit_price,
    price_tier,
    line_revenue,
    country
from {{ ref('stg_transactions') }}
"""

# ── fact_returns.sql ──────────────────────────────────────────
fact_returns = """select
    invoice_no,
    stock_code,
    description,
    qty_returned,
    invoice_date,
    unit_price,
    customer_id,
    country,
    revenue_lost
from {{ ref('stg_returns') }}
"""

# ── mart_funnel.sql ───────────────────────────────────────────
mart_funnel = """with sessions as (
    select
        customer_id,
        session_date,
        country,
        invoice_month,
        invoice_week,
        sum(line_revenue)           as basket_value,
        count(distinct invoice_no)  as nb_invoices,
        sum(quantity)               as nb_items,
        round(avg(invoice_hour), 0) as avg_hour,
        max(invoice_day_of_week)    as weekday,
        max(category_proxy)         as main_category,
        max(price_tier)             as dominant_price_tier
    from {{ ref('fact_transactions') }}
    group by customer_id, session_date, country,
             invoice_month, invoice_week
)
select
    *,
    true as is_converted,
    case
        when basket_value < 20  then 'Low (<20 GBP)'
        when basket_value < 100 then 'Medium (20-100)'
        else                         'High (>100)'
    end as basket_tier
from sessions
"""

# ── dim_customer.sql ──────────────────────────────────────────
dim_customer = """select
    customer_id,
    country,
    min(session_date)              as first_purchase_date,
    max(session_date)              as last_purchase_date,
    count(distinct session_date)   as nb_active_days,
    count(distinct invoice_no)     as total_orders,
    round(sum(line_revenue), 2)    as total_revenue,
    round(avg(line_revenue), 2)    as avg_line_revenue
from {{ ref('fact_transactions') }}
group by customer_id, country
"""

# ── dim_date.sql ──────────────────────────────────────────────
dim_date = """with spine as (
    select unnest(
        generate_series(
            date '2010-12-01',
            date '2011-12-31',
            interval '1 day'
        )
    ) as date_day
)
select
    date_day,
    year(date_day)                      as year,
    month(date_day)                     as month,
    week(date_day)                      as week_num,
    dayofweek(date_day)                 as weekday_num,
    dayname(date_day)                   as weekday_name,
    monthname(date_day)                 as month_name,
    month(date_day) in (11, 12)         as is_peak_season,
    case
        when strftime(date_day, '%m-%d') between '02-10' and '02-16'
            then 'Valentines'
        when strftime(date_day, '%m-%d') between '12-15' and '12-31'
            then 'Christmas'
        when strftime(date_day, '%m-%d') between '10-25' and '10-31'
            then 'Halloween'
        else null
    end                                 as commercial_event
from spine
"""

# Écriture sur Drive
marts = {
    "models/marts/fact_transactions.sql" : fact_transactions,
    "models/marts/fact_returns.sql"      : fact_returns,
    "models/marts/mart_funnel.sql"       : mart_funnel,
    "models/marts/dim_customer.sql"      : dim_customer,
    "models/marts/dim_date.sql"          : dim_date,
}
for path, content in marts.items():
    with open(f"{DBT_PATH}/{path}", "w") as f:
        f.write(content)
    print(f"Écrit : {path}")

Écrit : models/marts/fact_transactions.sql
Écrit : models/marts/fact_returns.sql
Écrit : models/marts/mart_funnel.sql
Écrit : models/marts/dim_customer.sql
Écrit : models/marts/dim_date.sql


In [8]:
# ── dim_payment.sql ───────────────────────────────────────────
dim_payment = """-- dim_payment : proxy via mode_of_shipment (dataset Shipping)
-- Le dataset UCI Online Retail ne contient pas de mode de paiement.
-- Hypothèse documentée DEC-PAYMENT-001 : on utilise mode_of_shipment
-- comme dimension de traitement de commande, corrélée au canal d'achat.
select
    mode_of_shipment                        as payment_mode_proxy,
    COUNT(*)                                as nb_orders,
    ROUND(AVG(discount_pct), 1)             as avg_discount_pct,
    ROUND(AVG(customer_rating), 2)          as avg_satisfaction,
    ROUND(AVG(estimated_shipping_cost), 2)  as avg_shipping_cost,
    ROUND(100.0 * SUM(CASE WHEN delivered_on_time = 1 THEN 1 ELSE 0 END)
          / COUNT(*), 1)                    as pct_on_time,
    -- enrichissement métier
    case mode_of_shipment
        when 'Ship'   then 'Maritime — délai long, coût faible'
        when 'Flight' then 'Aérien — délai court, coût élevé'
        when 'Road'   then 'Routier — délai moyen, coût moyen'
    end                                     as mode_description
from {{ ref('stg_shipping') }}
group by mode_of_shipment
"""

with open(f"{DBT_PATH}/models/marts/dim_payment.sql", "w") as f:
    f.write(dim_payment)
print("dim_payment.sql écrit")

dim_payment.sql écrit


## Cellule 7 — Écrire schema.yml (tests dbt)

In [9]:
schema_yml = """version: 2

models:
  - name: stg_transactions
    description: "Ventes nettes Silver filtrées — identifiées, sans retours, sans outliers"
    columns:
      - name: invoice_no
        tests: [not_null]
      - name: customer_id
        tests: [not_null]
      - name: line_revenue
        tests: [not_null]

  - name: fact_transactions
    description: "Table de faits centrale — 1 ligne = 1 ligne de vente nette"
    columns:
      - name: invoice_no
        tests: [not_null]
      - name: customer_id
        tests: [not_null]
      - name: category_proxy
        tests: [not_null]

  - name: mart_funnel
    description: "Sessions converties agrégées par CustomerID x jour — ST1"
    columns:
      - name: customer_id
        tests: [not_null]
      - name: session_date
        tests: [not_null]
      - name: basket_value
        tests: [not_null]
      - name: is_converted
        tests:
          - accepted_values:
              values: [true]

  - name: dim_customer
    description: "Dimension client avec historique agrégé"
    columns:
      - name: customer_id
        tests: [not_null, unique]

  - name: dim_date
    description: "Calendrier complet Dec 2010 — Dec 2011 avec flags UK"
    columns:
      - name: date_day
        tests: [not_null, unique]
  - name: dim_payment
    description: >
      Proxy dim_payment via mode_of_shipment (Shipping dataset).
      Le dataset UCI ne contient pas de mode de paiement réel —
      hypothèse documentée dans DECISIONS.md sous DEC-PAYMENT-001.
    columns:
      - name: payment_mode_proxy
        tests: [not_null, unique]
"""

with open(f"{DBT_PATH}/models/schema.yml", "w") as f:
    f.write(schema_yml)
print("schema.yml écrit")

# Vérification structure finale
print("\nStructure DBT_Gold créée :")
for root, dirs, files in os.walk(DBT_PATH):
    level = root.replace(DBT_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

schema.yml écrit

Structure DBT_Gold créée :
DBT_Gold/
  dbt_project.yml
  target/
    catalog.json
    partial_parse.msgpack
    index.html
    graph.gpickle
    graph_summary.json
    manifest.json
    semantic_manifest.json
    run_results.json
    run/
      ecommerce/
        models/
          staging/
            stg_shipping.sql
            stg_transactions.sql
            stg_returns.sql
          schema.yml/
            not_null_fact_transactions_invoice_no.sql
            not_null_dim_customer_customer_id.sql
            accepted_values_mart_rfm_m_score__1__2__3__4__5.sql
            not_null_dim_date_date_day.sql
            accepted_values_mart_clv_99b1edfd532f102639ae3e8c36294d12.sql
            accepted_values_mart_rfm_f_score__1__2__3__4__5.sql
            accepted_values_mart_rfm_r_score__1__2__3__4__5.sql
            not_null_fact_transactions_customer_id.sql
            not_null_mart_clv_clv_simple.sql
            accepted_values_mart_rfm_471de7decfe4044180ad21535fd95

## Cellule 8 — Lancer dbt debug, run, test

In [10]:
import os
os.chdir(DBT_PATH)
print("Dossier courant :", os.getcwd())

print("\n=== dbt debug ===")
!dbt debug --profiles-dir ~/.dbt

Dossier courant : /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/DBT_Gold

=== dbt debug ===
11:48:21  Running with dbt=1.11.11
11:48:21  dbt version: 1.11.11
11:48:21  python version: 3.12.13
11:48:21  python path: /usr/bin/python3
11:48:21  os info: Linux-6.6.122+-x86_64-with-glibc2.35
11:48:21  Using profiles dir at /root/.dbt
11:48:21  Using profiles.yml file at /root/.dbt/profiles.yml
11:48:21  Using dbt_project.yml file at /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/DBT_Gold/dbt_project.yml
11:48:21  adapter type: duckdb
11:48:21  adapter version: 1.10.1
11:48:22  Configuration:
11:48:22    profiles.yml file [OK found and valid]
11:48:22    dbt_project.yml file [OK found and valid]
11:48:22  Required dependencies:
11:48:22   - git [OK found]

11:48:22  Connection:
11:48:22    database: gold
11:48:22    schema: main
11:48:22    path: /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/gold.duckdb
11:48:22    config_options: None
11:48:22    extensions: None
11:48:22    

In [11]:
print("\n=== dbt run ===")
!dbt run --profiles-dir ~/.dbt


=== dbt run ===
11:48:29  Running with dbt=1.11.11
11:48:29  Registered adapter: duckdb=1.10.1
11:48:33  Found 12 models, 16 data tests, 485 macros
11:48:33  
11:48:33  Concurrency: 4 threads (target='dev')
11:48:33  
11:48:33  1 of 12 START sql table model main.dim_date .................................... [RUN]
11:48:33  2 of 12 START sql table model main.mart_clv .................................... [RUN]
11:48:33  3 of 12 START sql table model main.mart_cohorts ................................ [RUN]
11:48:33  4 of 12 START sql table model main.mart_rfm .................................... [RUN]
11:48:35  1 of 12 OK created sql table model main.dim_date ............................... [OK in 2.41s]
11:48:35  5 of 12 START sql view model main.stg_returns .................................. [RUN]
11:48:37  3 of 12 OK created sql table model main.mart_cohorts ........................... [OK in 3.66s]
11:48:37  6 of 12 START sql view model main.stg_shipping .............................

In [12]:
print("\n=== dbt test ===")
!dbt test --profiles-dir ~/.dbt


=== dbt test ===
11:48:49  Running with dbt=1.11.11
11:48:49  Registered adapter: duckdb=1.10.1
11:48:50  Found 12 models, 16 data tests, 485 macros
11:48:50  
11:48:50  Concurrency: 4 threads (target='dev')
11:48:50  
11:48:50  1 of 16 START test accepted_values_mart_funnel_is_converted__True .............. [RUN]
11:48:50  2 of 16 START test not_null_dim_customer_customer_id ........................... [RUN]
11:48:50  3 of 16 START test not_null_dim_date_date_day .................................. [RUN]
11:48:50  4 of 16 START test not_null_dim_payment_payment_mode_proxy ..................... [RUN]
11:48:52  4 of 16 PASS not_null_dim_payment_payment_mode_proxy ........................... [PASS in 1.53s]
11:48:52  2 of 16 PASS not_null_dim_customer_customer_id ................................. [PASS in 1.56s]
11:48:52  1 of 16 PASS accepted_values_mart_funnel_is_converted__True .................... [PASS in 1.58s]
11:48:52  3 of 16 PASS not_null_dim_date_date_day .....................

## Cellule 9 — Vérification : lire mart_funnel depuis DuckDB

In [13]:
import duckdb
import pandas as pd

con = duckdb.connect(GOLD_DB)

print("Tables disponibles dans gold.duckdb :")
print(con.execute("SHOW TABLES").df().to_string())

print("\n--- KPIs ST1 depuis mart_funnel ---")
kpis = con.execute("""
    SELECT
        COUNT(*)                               as nb_sessions_converties,
        COUNT(DISTINCT customer_id)            as nb_clients,
        ROUND(AVG(basket_value), 2)            as panier_moyen_GBP,
        ROUND(SUM(basket_value), 0)            as ca_total_GBP,
        -- Proxy abandon (benchmark 70%)
        ROUND(COUNT(*) / 0.30, 0)             as sessions_estimees_total,
        ROUND(COUNT(*) / 0.30 * 0.70, 0)     as abandons_estimes,
        ROUND(AVG(basket_value) * COUNT(*) / 0.30 * 0.70, 0) as manque_gagner_GBP
    FROM mart_funnel
""").df()
print(kpis.T.to_string())

print("\n--- Top 10 pays par CA ---")
print(con.execute("""
    SELECT country,
           COUNT(*)                as nb_sessions,
           ROUND(SUM(basket_value),0) as ca_gbp
    FROM mart_funnel
    GROUP BY country
    ORDER BY ca_gbp DESC LIMIT 10
""").df().to_string())

print("\n--- Répartition basket_tier ---")
print(con.execute("""
    SELECT basket_tier,
           COUNT(*) as nb,
           ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 1) as pct
    FROM mart_funnel GROUP BY basket_tier ORDER BY nb DESC
""").df().to_string())

Tables disponibles dans gold.duckdb :
                 name
0        dim_customer
1            dim_date
2         dim_payment
3        fact_returns
4   fact_transactions
5            mart_clv
6        mart_cohorts
7         mart_funnel
8            mart_rfm
9         stg_returns
10       stg_shipping
11   stg_transactions

--- KPIs ST1 depuis mart_funnel ---
                                   0
nb_sessions_converties      15963.00
nb_clients                   4245.00
panier_moyen_GBP              350.04
ca_total_GBP              5587720.00
sessions_estimees_total     53210.00
abandons_estimes            37247.00
manque_gagner_GBP        13038013.00

--- Top 10 pays par CA ---
          country  nb_sessions     ca_gbp
0  United Kingdom        14385  4745038.0
1         Germany          407   174530.0
2          France          348   150068.0
3            EIRE          175   149938.0
4     Switzerland           43    41927.0
5           Spain           83    40076.0
6     Netherlands    

## Cellule 10 — Générer la doc dbt
> Optionnel. Génère le lineage DAG interactif dans `DBT_Gold/target/`.

In [14]:
os.chdir(DBT_PATH)
!dbt docs generate --profiles-dir ~/.dbt
print("\nDoc générée dans DBT_Gold/target/")
print("Fichiers :", os.listdir(f"{DBT_PATH}/target"))

11:49:02  Running with dbt=1.11.11
11:49:03  Registered adapter: duckdb=1.10.1
11:49:04  Found 12 models, 16 data tests, 485 macros
11:49:04  
11:49:04  Concurrency: 4 threads (target='dev')
11:49:04  
11:49:04  Encountered an error:
IO Error: Could not set lock on file "/content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/gold.duckdb": Conflicting lock is held in /usr/bin/python3.12 (PID 750). See also https://duckdb.org/docs/stable/connect/concurrency
11:49:04  Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/dbt/adapters/duckdb/connections.py", line 95, in exception_handler
    yield
  File "/usr/local/lib/python3.12/dist-packages/dbt/adapters/sql/connections.py", line 147, in add_query
    cursor = connection.handle.cursor()
             ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/dbt/adapters/contracts/connection.py", line 96, in handle
    self._handle.resolve(self)
  File "/usr/local/lib/python3.12/dist-packages/dbt/ad

---
## Récapitulatif des tables Gold créées

| Table | Type | ST | Lignes attendues |
|---|---|---|---|
| `stg_transactions` | Vue staging | Tous | ~391 000 |
| `stg_returns` | Vue staging | ST1, ST4 | ~9 200 |
| `stg_shipping` | Vue staging | ST1, ST4 | ~11 000 |
| `fact_transactions` | Table | Tous | ~391 000 |
| `fact_returns` | Table | ST1, ST4 | ~9 200 |
| `mart_funnel` | Table | ST1 | ~19 000 sessions |
| `dim_customer` | Table | ST1, ST2 | ~4 300 clients |
| `dim_date` | Table | Tous | 396 jours |

**Étape suivante :** Notebook EDA ST1 — Funnel chart, heatmap heure×jour, carte choroplèthe.

In [16]:
# ══════════════════════════════════════════════════════════════
# Cellule 11 — Export de toutes les tables Gold en CSV
# ══════════════════════════════════════════════════════════════
import duckdb
import os

# Dossier de sortie CSV
CSV_PATH = f"{DATA_PATH}/gold/csv_exports"
os.makedirs(CSV_PATH, exist_ok=True)

con = duckdb.connect(GOLD_DB)

# Récupérer la liste de toutes les tables
tables = con.execute("SHOW TABLES").df()["name"].tolist()
print(f"Tables trouvées : {tables}\n")

# Exporter chaque table en CSV
for table in tables:
    output_file = f"{CSV_PATH}/{table}.csv"
    con.execute(f"""
        COPY {table} TO '{output_file}'
        (HEADER, DELIMITER ',')
    """)
    row_count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}.csv exporté ({row_count:,} lignes) → {output_file}")

con.close()
print(f"\n Tous les CSV sont dans : {CSV_PATH}")
print(f"Fichiers : {os.listdir(CSV_PATH)}")

Tables trouvées : ['dim_customer', 'dim_date', 'dim_payment', 'fact_returns', 'fact_transactions', 'mart_clv', 'mart_cohorts', 'mart_funnel', 'mart_rfm', 'stg_returns', 'stg_shipping', 'stg_transactions']

dim_customer.csv exporté (4,253 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/dim_customer.csv
dim_date.csv exporté (396 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/dim_date.csv
dim_payment.csv exporté (3 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/dim_payment.csv
fact_returns.csv exporté (8,872 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/fact_returns.csv
fact_transactions.csv exporté (372,684 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/fact_transactions.csv
mart_clv.csv exporté (4,253 lignes) → /content/drive/MyDrive/Projet_RFM_CLV_Ecommerce/Data/gold/csv_exports/mart_clv.csv
mart_cohorts.csv exporté (91 lig